# Quick Start: From Raw Case Reports to Structured Patient Records

Discover a schema, extract records, and normalize clinical terminology — on real rare disease case reports from PubMed Central.

**What you'll learn:** How catchfly automates the entire discovery → extraction → field selection → normalization pipeline on unstructured medical text.

**Dataset:** [RareArena](https://huggingface.co/datasets/THUMedInfo/RareArena) — rare disease case reports mapped to Orphanet.

**Estimated API cost:** ~$0.01

In [ ]:
# pip install "catchfly[openai,export]" datasets
# export OPENAI_API_KEY="sk-..."

## Load 50 real case reports

In [ ]:
import json

from huggingface_hub import hf_hub_download

from catchfly import Document, Pipeline
from catchfly.discovery import SinglePassDiscovery
from catchfly.extraction import LLMDirectExtraction
from catchfly.normalization import LLMCanonicalization
from catchfly.selection import LLMFieldSelector

path = hf_hub_download(repo_id="THUMedInfo/RareArena", filename="RDC.json", repo_type="dataset")
raw = [json.loads(line) for line in open(path) if line.strip()]

docs = []
for row in raw:
    case = row.get("case_report", "") or ""
    tests = row.get("test_results", "") or ""
    if len(case) < 100:
        continue
    full_text = case + (f"\n\nTest results: {tests}" if tests else "")
    docs.append(Document(content=full_text, id=row.get("_id", f"case-{len(docs)}")))
    if len(docs) >= 50:
        break

print(f"{len(docs)} case reports loaded (avg {sum(len(d.content) for d in docs) // len(docs)} chars)")
print(f"\nSample:\n{docs[0].content[:300]}...")

## Run the pipeline

No hand-crafted schema. No field list. Catchfly discovers the structure, extracts records, decides what to normalize, and normalizes it.

In [ ]:
pipeline = Pipeline(
    discovery=SinglePassDiscovery(model="gpt-5.4-mini", max_fields=8),
    extraction=LLMDirectExtraction(model="gpt-5.4-mini"),
    normalization=LLMCanonicalization(model="gpt-5.4-mini"),
    field_selector=LLMFieldSelector(model="gpt-5.4-mini"),
)

results = pipeline.run(
    docs, domain_hint="Rare disease clinical case reports with symptoms, diagnosis, and treatment"
)

In [ ]:
df = results.to_dataframe()
df.head(10)

That's it. From raw clinical narratives, catchfly:
1. **Discovered** a schema (patient_age, symptoms, diagnosis, treatment, ...)
2. **Extracted** structured records from 50 case reports
3. **Auto-selected** which fields to normalize (symptoms, patient_sex — not free-text like diagnosis)
4. **Normalized** clinical terminology — grouping "dyspnea", "shortness of breath", and "exertional dyspnea" into a single canonical form

No `normalize_fields` parameter needed. The `LLMFieldSelector` decided.

## What happened under the hood

### 1. Schema discovery

Catchfly sampled 5 documents and inferred a JSON Schema:

In [ ]:
print(json.dumps(results.schema.json_schema, indent=2))

### 2. Extracted records

In [ ]:
for record in results.records[:5]:
    symptoms = getattr(record, "symptoms", []) or []
    print(f"{record.diagnosis}")
    print(f"  age={record.patient_age}, sex={record.patient_sex}")
    print(f"  symptoms={symptoms[:3]}")
    print()

### 3. Field selection + normalization

The `LLMFieldSelector` chose which fields to normalize. Then `LLMCanonicalization` grouped the variants:

In [ ]:
print("Auto-selected fields:", list(results.normalizations.keys()))
print()

for field_name, norm in results.normalizations.items():
    n_multi = sum(1 for m in norm.clusters.values() if len(m) > 1)
    print(f"'{field_name}' — {len(norm.clusters)} canonical groups ({n_multi} with merged variants):")
    for canonical, members in sorted(norm.clusters.items(), key=lambda x: -len(x[1])):
        if len(members) > 1:
            print(f"  {canonical}: {members[:5]}")
    print()

### 4. Overriding field selection

Auto-selection works well, but you can always override it:

```python
results = pipeline.run(docs, normalize_fields=["symptoms"])  # only normalize symptoms
results = pipeline.run(docs, normalize_fields="all")          # normalize all string fields
```

### 5. Cost report

In [ ]:
print(f"Total cost: ${results.report.total_cost_usd:.4f}")
print(f"Total tokens: {results.report.total_input_tokens + results.report.total_output_tokens:,}")

## What makes this different

No other open-source library takes raw clinical narratives and produces structured, normalized patient records in a single pipeline. Catchfly handles the entire workflow:

| Step | What catchfly did | You wrote |
|------|-------------------|-----------|
| Schema discovery | Inferred 8 fields from 5 sample docs | Nothing |
| Extraction | Parsed 50 case reports into typed records | Nothing |
| Field selection | Chose symptoms + patient_sex for normalization | Nothing |
| Normalization | Grouped 197 symptom variants into 67 canonical forms | Nothing |

## Next steps

- [02_rare_disease.ipynb](02_rare_disease.ipynb) — Full pipeline with HPO ontology coding and CascadeNormalization
- [03_product_catalog.ipynb](03_product_catalog.ipynb) — Cost control, checkpoints, per-field normalization strategies
- [04_custom_schema.ipynb](04_custom_schema.ipynb) — Skip discovery, bring your own Pydantic models
- [05_local_models.ipynb](05_local_models.ipynb) — Run everything locally with Ollama